# Reranker Evaluation

Notebook for explicit evaluation of the cross-encoder reranker on top of the first-stage retriever.


## Pipeline Overview

Stages:
1. setup runtime and imports
2. load and preprocess data
3. prepare first-stage retriever
4. predict categories
5. build the cross-encoder reranker
6. run base retrieval on train queries
7. rerank those results
8. compare metrics and inspect bad cases


In [1]:
import sys
from pathlib import Path

def detect_runtime_environment() -> str:
    try:
        import google.colab  # type: ignore  # noqa: F401
        return 'colab'
    except Exception:
        if Path('/kaggle/input').exists():
            return 'kaggle'
        return 'local'

def add_project_root_to_syspath(project_name: str = 'retrieval_project') -> None:
    runtime_env = detect_runtime_environment()
    candidates = [Path.cwd(), *Path.cwd().parents]
    if runtime_env == 'colab':
        drive_root = Path('/content/drive/MyDrive')
        if not drive_root.exists():
            from google.colab import drive  # type: ignore
            drive.mount('/content/drive', force_remount=False)
        candidates = [Path('/content'), Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives'), *candidates]
    elif runtime_env == 'kaggle':
        candidates = [Path('/kaggle/working'), *candidates]

    seen = set()
    for base in candidates:
        key = str(base)
        if key in seen:
            continue
        seen.add(key)
        if (base / 'src' / 'infra' / 'notebook.py').exists():
            sys.path.insert(0, str(base))
            return
        if (base / project_name / 'src' / 'infra' / 'notebook.py').exists():
            sys.path.insert(0, str(base / project_name))
            return
        if runtime_env == 'colab' and base.exists():
            for match in base.rglob(project_name):
                if (match / 'src' / 'infra' / 'notebook.py').exists():
                    sys.path.insert(0, str(match))
                    return
    raise FileNotFoundError('Could not locate project root containing src/infra/notebook.py')

add_project_root_to_syspath()

import pandas as pd

from src.infra.notebook import setup_notebook
runtime_env, project_root = setup_notebook()
print(f'Detected runtime: {runtime_env}')
print(f'Project root    : {project_root}')


Detected runtime: local
Project root    : /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project


## Step 1: Load Data

Load the normalized train/test views and the training ground-truth labels.


In [2]:
from src.config import DEFAULT_CONFIG
from src.evaluation import leaderboard_score, load_ground_truth, mrr_at_k
from src.output.diagnostics import build_bad_case_entry, build_scalar_map
from src.pipeline import (
    bootstrap,
    build_cross_encoder_reranker,
    load_project_frames,
    predict_categories,
    prepare_retrievers,
    rerank_retrieval_results,
    run_first_stage_retrieval,
)
from src.reranking import rerank_results_with_cross_encoder
from src.retrieval import truncate_results

paths, config = bootstrap()
frames = load_project_frames(paths, DEFAULT_CONFIG)
ground_truth = load_ground_truth(paths.data_dir / 'qgts_train.json')


## Step 2: Prepare First-Stage Retrieval and Categories

Prepare the configured retriever and the category predictions needed for reranking and category-aware analysis.


In [3]:
prepared_retrievers = prepare_retrievers(frames, paths, config=config)
category_artifacts = predict_categories(frames, paths, ground_truth=ground_truth, config=config)
print(f'Category accuracy: {category_artifacts.classifier_accuracy:.5f}')


Loading model weights from cache: /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project/cache/sentence_transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading docs embeddings from cache: docs_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_6485efc872cab480.npy


Loading category classifier from cache: category_classifier_e3f41fe8a612f7c0.pkl


Category accuracy: 0.92661


## Step 3: Build Cross-Encoder

Load or train the cross-encoder reranker using the training labels.


In [4]:
cross_encoder_reranker = build_cross_encoder_reranker(frames, paths, ground_truth, config=config)
if cross_encoder_reranker is None:
    raise ValueError('Cross-encoder reranking is disabled in config.')
print('Cross-encoder reranker is ready.')


Loading cross-encoder from cache: cross-encoder_ms-marco-MiniLM-L6-v2_c2a686d8799ae524


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-encoder reranker is ready.


## Step 4: Base Retrieval and Reranking

Run the first-stage retriever on the training queries, then apply cross-encoder reranking to the same candidate lists.


In [5]:
eval_top_k = max(config.retrieval_pipeline.evaluation_top_ks)
base_results, _ = run_first_stage_retrieval(
    frames=frames,
    paths=paths,
    prepared_retrievers=prepared_retrievers,
    category_artifacts=category_artifacts,
    split='train',
    top_k=eval_top_k,
    config=config,
)
reranked_results, rerank_diagnostics = rerank_results_with_cross_encoder(
    results=base_results,
    query_frame=frames.train_queries,
    docs_frame=frames.docs,
    cross_encoder=cross_encoder_reranker,
    query_category_map=category_artifacts.train_query_category_map,
    doc_category_map=category_artifacts.doc_category_map,
    infer_batch_size=config.cross_encoder.infer_batch_size,
    rerank_top_m=config.cross_encoder.rerank_top_m,
    category_bonus=config.cross_encoder.category_bonus if config.retrieval_pipeline.enable_category_filter else 0.0,
    return_diagnostics=True,
)


Starting category-filtered retrieval: model=embedding, top_k=12,500, queries=327, predicted_categories=5


Loading docs embeddings from cache: docs_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_f2ec1b95e73064f5.npy
Starting retrieval: model=embedding
  parameters: top_k=12,500, docs=22,998, queries=33, prepared_artifacts=yes, embedding_kind='queries_train_android_filtered'
Loading queries_train_android_filtered embeddings from cache: queries_train_android_filtered_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_e0dbe51a7887ed9f.npy
  [Embedding] scoring 33 queries against 22,998 docs with capped_top_k=12,500, chunk_size=32, embedding_cache_key='queries_train_android_filtered'
  [Embedding] chunk 1/2: queries 1-32
  [Embedding] chunk 2/2: queries 33-33
Completed retrieval: model=embedding, results=33 queries, elapsed=0.0s


Loading docs embeddings from cache: docs_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_5c02b9494fdb73f4.npy
Starting retrieval: model=embedding
  parameters: top_k=12,500, docs=45,301, queries=41, prepared_artifacts=yes, embedding_kind='queries_train_gaming_filtered'
Loading queries_train_gaming_filtered embeddings from cache: queries_train_gaming_filtered_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_59a0c0a2a237034e.npy
  [Embedding] scoring 41 queries against 45,301 docs with capped_top_k=12,500, chunk_size=32, embedding_cache_key='queries_train_gaming_filtered'
  [Embedding] chunk 1/2: queries 1-32
  [Embedding] chunk 2/2: queries 33-41
Completed retrieval: model=embedding, results=41 queries, elapsed=0.1s


Loading docs embeddings from cache: docs_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_a97f454119668600.npy
Starting retrieval: model=embedding
  parameters: top_k=12,500, docs=32,176, queries=58, prepared_artifacts=yes, embedding_kind='queries_train_programmers_filtered'
Loading queries_train_programmers_filtered embeddings from cache: queries_train_programmers_filtered_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_202fa36fcdb4ba24.npy
  [Embedding] scoring 58 queries against 32,176 docs with capped_top_k=12,500, chunk_size=32, embedding_cache_key='queries_train_programmers_filtered'
  [Embedding] chunk 1/2: queries 1-32
  [Embedding] chunk 2/2: queries 33-58
Completed retrieval: model=embedding, results=58 queries, elapsed=0.1s


Loading docs embeddings from cache: docs_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_ee362720a9bb0089.npy
Starting retrieval: model=embedding
  parameters: top_k=12,500, docs=68,184, queries=130, prepared_artifacts=yes, embedding_kind='queries_train_tex_filtered'
Loading queries_train_tex_filtered embeddings from cache: queries_train_tex_filtered_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_b060d44e3810208f.npy
  [Embedding] scoring 130 queries against 68,184 docs with capped_top_k=12,500, chunk_size=32, embedding_cache_key='queries_train_tex_filtered'
  [Embedding] chunk 1/5: queries 1-32
  [Embedding] chunk 2/5: queries 33-64
  [Embedding] chunk 3/5: queries 65-96
  [Embedding] chunk 4/5: queries 97-128


  [Embedding] chunk 5/5: queries 129-130
Completed retrieval: model=embedding, results=130 queries, elapsed=0.2s


Loading docs embeddings from cache: docs_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_dd2e6397ae8746fb.npy
Starting retrieval: model=embedding
  parameters: top_k=12,500, docs=47,382, queries=65, prepared_artifacts=yes, embedding_kind='queries_train_unix_filtered'
Loading queries_train_unix_filtered embeddings from cache: queries_train_unix_filtered_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_242721c1fa5c420e.npy
  [Embedding] scoring 65 queries against 47,382 docs with capped_top_k=12,500, chunk_size=32, embedding_cache_key='queries_train_unix_filtered'
  [Embedding] chunk 1/3: queries 1-32
  [Embedding] chunk 2/3: queries 33-64
  [Embedding] chunk 3/3: queries 65-65
Completed retrieval: model=embedding, results=65 queries, elapsed=0.1s


  [CrossEncoder] reranked 327/327 queries (top_m=45, category_bonus=0.50, total_pairs=14,715)


## Step 5: Compare Metrics

Measure how reranking changes Recall, Precision, MRR, Accuracy, and leaderboard score.


In [6]:
metrics_rows = []
for top_k in config.retrieval_pipeline.evaluation_top_ks:
    base_metrics = leaderboard_score(truncate_results(base_results, top_k), ground_truth, k=top_k, accuracy_value=category_artifacts.classifier_accuracy)
    reranked_metrics = leaderboard_score(truncate_results(reranked_results, top_k), ground_truth, k=top_k, accuracy_value=category_artifacts.classifier_accuracy)
    metrics_rows.append({'Variant': 'base', 'TopK': int(top_k), **base_metrics})
    metrics_rows.append({'Variant': 'reranked', 'TopK': int(top_k), **reranked_metrics})

metrics_df = pd.DataFrame(metrics_rows)
metrics_df


,Variant,TopK,Recall,Precision,MRR,Accuracy,LeaderboardScore
0,base,12500,0.905954,0.000657,0.442439,0.926606,0.568914
1,reranked,12500,0.905954,0.000657,0.601620,0.926606,0.608709


In [7]:
pivot = metrics_df.pivot(index='TopK', columns='Variant', values=['Recall', 'Precision', 'MRR', 'Accuracy', 'LeaderboardScore'])
pivot


Recall           Precision                 MRR           Accuracy  \
Variant      base  reranked      base  reranked      base reranked      base   
TopK                                                                           
12500    0.905954  0.905954  0.000657  0.000657  0.442439  0.60162  0.926606   

                  LeaderboardScore            
Variant  reranked             base  reranked  
TopK                                          
12500    0.926606         0.568914  0.608709

## Step 6: Inspect Bad Cases

Build a small set of reranker regressions so failures are easy to inspect in the notebook.


In [8]:
query_title_map = build_scalar_map(frames.train_queries_raw, 'title')
query_text_map = build_scalar_map(frames.train_queries_raw, 'text')
rerank_diag_by_query = {str(item['query_id']): item for item in rerank_diagnostics}
bad_case_candidates = []
for base_item, reranked_item in zip(base_results, reranked_results):
    query_id = str(base_item['query_id'])
    base_mrr = mrr_at_k([base_item], ground_truth, k=eval_top_k)
    reranked_mrr = mrr_at_k([reranked_item], ground_truth, k=eval_top_k)
    if reranked_mrr < base_mrr:
        bad_case_candidates.append({
            'query_id': query_id,
            'true_category': ground_truth[query_id].get('category'),
            'predicted_category': category_artifacts.train_query_category_map.get(query_id),
            'category_correct': ground_truth[query_id].get('category') == category_artifacts.train_query_category_map.get(query_id),
            'top_k': eval_top_k,
            'relevant_doc_ids': sorted(ground_truth[query_id]['relevant_doc_ids']),
            'retrieved_doc_ids': list(reranked_item['relevant_docs']),
            'base_retrieved_doc_ids': list(base_item['relevant_docs']),
            'hit_count': sum(doc_id in ground_truth[query_id]['relevant_doc_ids'] for doc_id in reranked_item['relevant_docs']),
            'base_hit_count': sum(doc_id in ground_truth[query_id]['relevant_doc_ids'] for doc_id in base_item['relevant_docs']),
            'first_relevant_rank': next((rank for rank, doc_id in enumerate(reranked_item['relevant_docs'], start=1) if doc_id in ground_truth[query_id]['relevant_doc_ids']), None),
            'base_first_relevant_rank': next((rank for rank, doc_id in enumerate(base_item['relevant_docs'], start=1) if doc_id in ground_truth[query_id]['relevant_doc_ids']), None),
            'delta_hit_count': sum(doc_id in ground_truth[query_id]['relevant_doc_ids'] for doc_id in reranked_item['relevant_docs']) - sum(doc_id in ground_truth[query_id]['relevant_doc_ids'] for doc_id in base_item['relevant_docs']),
            'delta_recall_at_k': 0.0,
            'delta_precision_at_k': 0.0,
            'delta_reciprocal_rank': reranked_mrr - base_mrr,
            'rerank_diagnostics': rerank_diag_by_query.get(query_id),
        })

bad_cases = [
    build_bad_case_entry(item, query_title_map=query_title_map, query_text_map=query_text_map, doc_category_map=category_artifacts.doc_category_map)
    for item in sorted(bad_case_candidates, key=lambda row: row['delta_reciprocal_rank'])[:10]
]
bad_cases[:3]


[{'query': {'query_id': 'e31a4ec1-b357-4145-9933-8c13a7a9bd47_76162',
   'title': '',
   'text_preview': 'How do I capture the return status and use tee at the same time in korn shell?',
   'true_category': 'unix',
   'predicted_category': 'unix',
   'category_correct': True},
  'failure': {'delta_hit_count': 0,
   'delta_recall_at_k': 0.0,
   'delta_precision_at_k': 0.0,
   'delta_reciprocal_rank': -0.9666666666666667,
   'base_first_relevant_rank': 1,
   'reranked_first_relevant_rank': 30,
   'base_hit_count': 6,
   'reranked_hit_count': 6,
   'failure_reasons': ['first relevant moved from rank 1 to rank 30',
    "non-relevant document promoted: '43d78a6c-7cd2-465e-8c50-e7bbe1c7d23b_71492'",
    "relevant document demoted: '77a43ee6-da88-4e97-a90a-0b77a5fc27c6_96308'"]},
  'reranker_context': {'reranked': True,
   'rerank_top_m': 45,
   'category_bonus': 0.5,
   'predicted_query_category': 'unix',
   'candidate_doc_count': 45},
  'relevant_docs': [{'doc_id': '093f0d60-ce52-435e-93a9-